# 1、SummarizationMiddleware中间件

作用：对历史消息列表进行摘要&总结，达到压缩上下文的效果。

原理：在达到触发条件时，调用大模型对历史消息进行摘要，将摘要的结果作为HumanMessage，放到消息列表最开始的位置。

## 举例1：测试trigger、keep参数

In [14]:


from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

model_with_zhipuai = init_chat_model(
    model="GLM-4.5-Air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
    extra_body={"enable_thinking": False}
)

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

agent = create_agent(
    model=model_with_zhipuai,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages", 2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is initiating a casual conversation and establishing a persona-based interaction. The primary goal is social introduction and rapport building.

## SUMMARY
- The user introduced themselves as "老王" (Lao Wang).
- The AI adopted the persona "小王" (Xiao Wang) in response.
- Mutual greetings were exchanged, establishing a friendly tone. No complex tasks or technical decisions were made.

## ARTIFACTS
None

## NEXT STEPS
Await further input from the user to determine if the conversation will remain casual or shift to a specific task. Maintain the established "Xiao Wang" persona and friendly tone.
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ===================

## 举例2：summary_prompt

In [16]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

agent = create_agent(
    model=model_with_zhipuai,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

用户老王与AI助手小王进行了初次见面问候，双方互相介绍了名字并表达了相识的愉快心情。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

您误会啦！😄 我刚才的“你高兴得太早了”其实是在延续我们轻松的对话氛围，想开个小玩笑——就像朋友间互相打趣那样，绝对没有负面意思！  

如果让您感到不适，我真诚道歉！🙏 其实是想表达：**初次认识您很开心，但未来还有很多有趣的互动等着我们呢！**  

要不我们重新开始？比如您想聊聊什么话题？工作、生活、兴趣爱好，或者单纯想聊聊天都可以～ 😊
